# 🎙️ AI-Powered Multi-Agent Call Center

An intelligent voice-enabled call center system with specialized AI agents for sales, tech support, billing, and appointments.

## ✨ Features

- **4 Specialized AI Agents** with unique voices and personalities
  - Sarah (Sales) - Professional Female voice
  - Bindu (Tech Support) - Calm Female voice
  - Beth (Billing) - Energetic Female voice
  - Alex (Appointments) - Friendly Male voice

- **Intelligent Call Routing** - AI-powered intent classification
- **Seamless Transfers** - Agents transfer calls automatically when needed
- **Voice Interaction** - Speech-to-text and realistic TTS with Cartesia
- **Conversation Tracking** - Call logs, statistics, and analytics

## 🚀 Quick Start

1. Install required packages (Cell 2)
2. Add your API keys (Cell 4)
3. Run setup cells (5-24)
4. Start voice conversation (Cell 38)

## 📋 Requirements

- Python 3.8+
- Microphone for voice input
- Internet connection for speech recognition
- API keys: Cerebras AI, Cartesia TTS

---

In [1]:
# Install all required packages for our voice sales agent
# Note: We're skipping silero due to Windows DLL compatibility issues
!pip install livekit-agents livekit-plugins-openai livekit-plugins-cartesia -q
!pip install openai speechrecognition pyttsx3 -q

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


## Cell 2: Configure API Keys

In [ ]:
import os
from pathlib import Path

# API Keys Configuration
# IMPORTANT: Replace these with your own API keys
# Never commit real API keys to GitHub!

CARTESIA_API_KEY = "your_cartesia_api_key_here"  # Get from https://cartesia.ai
CEREBRAS_API_KEY = "your_cerebras_api_key_here"  # Get from https://cerebras.ai

# Optional: OpenAI API Key if you want to use GPT instead of Cerebras
OPENAI_API_KEY = "your_openai_api_key_here"  # Get from https://platform.openai.com/api-keys

# Optional: LiveKit credentials if using LiveKit features
LIVEKIT_API_KEY = "your_livekit_api_key"
LIVEKIT_API_SECRET = "your_livekit_secret"
LIVEKIT_URL = "your_livekit_url"

# Set all API keys in environment variables
os.environ["CARTESIA_API_KEY"] = CARTESIA_API_KEY
os.environ["CEREBRAS_API_KEY"] = CEREBRAS_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("✅ API keys configured")
print("")
print("⚠️ IMPORTANT: Before running this notebook:")
print("   1. Replace API keys above with your own")
print("   2. Never commit real API keys to version control")
print("   3. Consider using environment variables or .env file")

✅ Libraries imported and API keys configured
🔑 IMPORTANT: Replace OPENAI_API_KEY with your actual OpenAI API key!
   Get one at: https://platform.openai.com/api-keys


## Cell 3: Download Product Catalog

In [3]:
# Download sample Product JSON Data (Windows-compatible)
import urllib.request
import json
from pathlib import Path

# Create context directory
context_dir = Path("context")
context_dir.mkdir(exist_ok=True)

# Download products.json
url = "https://gist.githubusercontent.com/ShayneP/f373c26c5166d90446f2bc08baf9bf46/raw/products.json"
products_file = context_dir / "products.json"

if not products_file.exists():
    print(f"📥 Downloading products.json...")
    urllib.request.urlretrieve(url, products_file)
    print(f"✅ Downloaded to {products_file}")
else:
    print(f"✅ products.json already exists at {products_file}")

# Load and display the products
with open(products_file, 'r') as f:
    products = json.load(f)
    
    # Handle both list and dict structures
    if isinstance(products, dict):
        print(f"\n📦 Loaded product catalog with {len(products)} items")
        # Show first 3 items from dictionary
        for i, (key, product) in enumerate(list(products.items())[:3]):
            name = product.get('name', key) if isinstance(product, dict) else str(product)
            print(f"  - {name}")
    elif isinstance(products, list):
        print(f"\n📦 Loaded {len(products)} products:")
        for product in products[:3]:
            print(f"  - {product.get('name', 'Unknown')}")
    else:
        print(f"\n📦 Loaded products: {type(products)}")

✅ products.json already exists at context\products.json

📦 Loaded product catalog with 3 items
  - [{'name': 'Premium Software Suite', 'description': 'All-in-one business management platform', 'key_benefits': ['Increases productivity by 40%', 'Reduces operational costs', 'Seamless integration'], 'price_range': '$99-299/month', 'target_customers': 'SMB, Enterprise'}]
  - pricing
  - objection_handlers


## Cell 4: Load Context Function

In [4]:
def load_context():
    """Load all files from context directory"""
    context_dir = Path("context")
    context_dir.mkdir(exist_ok=True)

    all_content = ""
    for file_path in context_dir.glob("*"):
        if file_path.is_file():
            try:
                content = file_path.read_text(encoding='utf-8')
                all_content += f"\n=== {file_path.name} ===\n{content}\n"
            except:
                pass

    return all_content.strip() or "No files found"

print(load_context())
print("✅ Context loading function ready")

=== products.json ===
{
  "products": [
    {
      "name": "Premium Software Suite",
      "description": "All-in-one business management platform",
      "key_benefits": [
        "Increases productivity by 40%",
        "Reduces operational costs",
        "Seamless integration"
      ],
      "price_range": "$99-299/month",
      "target_customers": "SMB, Enterprise"
    }
  ],
  "pricing": {
    "starter": {
      "price": 99,
      "features": [
        "Basic CRM",
        "Email integration"
      ]
    },
    "professional": {
      "price": 199,
      "features": [
        "Advanced analytics",
        "Custom workflows"
      ]
    },
    "enterprise": {
      "price": 299,
      "features": [
        "White-label",
        "API access",
        "Dedicated support"
      ]
    }
  },
  "objection_handlers": {
    "too_expensive": "I understand cost is important. Let's look at the ROI - most clients see 3x return in the first 6 months.",
    "need_time_to_think": "I appreciat

## Cell 5: Sales Agent with Product Context

In [8]:
# Enhanced Sales Agent with Product Context
import openai
from datetime import datetime

class SalesAgentWithContext:
    def __init__(self, api_key, use_cerebras=False):
        """Initialize the sales agent with OpenAI or Cerebras API and product context"""
        if use_cerebras:
            # Configure for Cerebras
            self.client = openai.OpenAI(
                api_key=api_key,
                base_url="https://api.cerebras.ai/v1"
            )
            self.model = "llama3.1-8b"  # Updated to correct Cerebras model name
            print("🧠 Using Cerebras API (Llama 3.1 8B)")
        else:
            # Configure for OpenAI
            self.client = openai.OpenAI(api_key=api_key)
            self.model = "gpt-3.5-turbo"
            print("🤖 Using OpenAI API (GPT-3.5 Turbo)")
        
        self.conversation_history = []
        self.lead_info = {}
        
        # Load context from files
        context = load_context()
        
        # Create system prompt with product context
        self.system_prompt = f"""
        You are a professional AI sales agent. You communicate naturally by voice.
        
        AVAILABLE PRODUCTS AND INFORMATION:
        {context}
        
        YOUR ROLE:
        - Help customers find the right products from the catalog above
        - Answer questions about features, pricing, and availability
        - Qualify leads by understanding their needs
        - Guide them through the sales process
        - Only use information from the product catalog provided
        
        COMMUNICATION STYLE:
        - Be friendly, professional, and helpful
        - Keep responses concise and natural (you're speaking, not writing)
        - Don't use bullets, special characters, or formatted lists
        - Ask clarifying questions to understand customer needs
        
        IMPORTANT: Only reference products and information from the context above.
        If asked about something not in the catalog, politely say you don't have that information.
        """
        
        print(f"✅ Sales Agent initialized with {len(context)} characters of product context")
    
    def chat(self, user_message):
        """Send a message and get a response"""
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system_prompt},
                *self.conversation_history
            ],
            temperature=0.7,
            max_tokens=200
        )
        
        assistant_message = response.choices[0].message.content
        self.conversation_history.append({
            "role": "assistant",
            "content": assistant_message
        })
        
        return assistant_message
    
    def reset_conversation(self):
        """Start a new conversation"""
        self.conversation_history = []
        print("🔄 Conversation reset")

# Initialize the agent - using Cerebras since that key is available
sales_agent = SalesAgentWithContext(api_key=CEREBRAS_API_KEY, use_cerebras=True)
print("💬 Ready to chat! Use: sales_agent.chat('your message')")

🧠 Using Cerebras API (Llama 3.1 8B)
✅ Sales Agent initialized with 1205 characters of product context
💬 Ready to chat! Use: sales_agent.chat('your message')


## Cell 6: Test Text-Based Conversation (Optional)

In [6]:
# Test the sales agent with the product catalog
print("🤖 Starting sales conversation with product context...\n")

# Greeting
response = sales_agent.chat("Hello! I'm looking for some tech products.")
print(f"Agent: {response}\n")

# Ask about a specific product
response = sales_agent.chat("What laptops do you have available?")
print(f"Agent: {response}\n")

# Ask for details
response = sales_agent.chat("Tell me about the most expensive laptop you have")
print(f"Agent: {response}\n")

print("✅ Conversation test complete!")
print("\n💡 Continue chatting: sales_agent.chat('your question')")
print("🔄 Reset conversation: sales_agent.reset_conversation()")

🤖 Starting sales conversation with product context...

Agent: Hello! It's great to hear from you. We have a variety of tech products that can help you streamline your business operations. Can you please tell me a bit more about what you're looking for? Are you looking for something to help with customer management, or maybe a tool to increase productivity?

Agent: I'm afraid we don't have any laptops for sale. Our focus is on software solutions that can help businesses like yours. We do have a Premium Software Suite that can integrate with various devices, but we don't carry any hardware products, including laptops. Would you like to know more about our software offerings?

Agent: I think there's still a misunderstanding. As I mentioned earlier, we don't carry any laptops. Our product catalog is focused on software solutions. If you're interested in learning more about our Premium Software Suite, I'd be happy to walk you through its features and benefits. It's a comprehensive all-in-on

## Cell 7: Voice-Enabled Sales Agent

This is the main voice agent class with:
- Speech recognition (microphone input)
- Text-to-speech (voice output with number pronunciation)
- Voice Activity Detection (VAD)
- Product context integration

In [13]:
# Enhanced Voice Agent with Cartesia TTS for realistic voice
import requests
from io import BytesIO
import pygame
import speech_recognition as sr

class CartesiaVoiceAgent(SalesAgentWithContext):
    def __init__(self, api_key, cartesia_key, use_cerebras=True):
        """Initialize voice agent with Cartesia TTS for ultra-realistic voice"""
        super().__init__(api_key, use_cerebras)
        
        # Initialize speech recognizer
        self.recognizer = sr.Recognizer()
        self.microphone = sr.Microphone()
        
        # Cartesia configuration
        self.cartesia_key = cartesia_key
        self.cartesia_url = "https://api.cartesia.ai/tts/bytes"
        
        # Available Cartesia voices - choose one!
        self.voices = {
            "professional_female": "694f9389-aac1-45b6-b726-9d9369183238",  # Confident female
            "friendly_male": "79a125e8-cd45-4c13-8a67-188112f4dd22",       # Warm male
            "energetic_female": "f114a467-c40a-4db8-964d-aaba89cd08fa",    # Upbeat female
            "calm_male": "a167e0f3-df7e-4d52-a9c3-f949145efdab",           # Soothing male
        }
        
        # Select default voice (change as needed)
        self.current_voice = "professional_female"
        
        # Configure VAD settings
        self.recognizer.energy_threshold = 1000
        self.recognizer.dynamic_energy_threshold = True
        self.recognizer.pause_threshold = 1.0
        self.recognizer.phrase_threshold = 0.3
        self.recognizer.non_speaking_duration = 0.8
        
        # Initialize pygame for audio playback
        try:
            pygame.mixer.init()
            print("✅ Pygame audio initialized")
        except Exception as e:
            print(f"⚠️ Pygame initialization error: {e}")
        
        print("🎭 Using Cartesia TTS - Ultra-realistic voice cloning")
        print(f"🎤 Current voice: {self.current_voice}")
        
        # Calibrate microphone
        print("\n🎤 Calibrating microphone (please be quiet for 3 seconds)...")
        with self.microphone as source:
            self.recognizer.adjust_for_ambient_noise(source, duration=3)
            print(f"✅ Calibration complete! Energy threshold: {self.recognizer.energy_threshold}")
        
        print("\n✅ Enhanced voice agent ready!")
        print(f"\n⚙️ Available voices: {', '.join(self.voices.keys())}")
        print("💡 Change voice: agent.change_voice('voice_name')")
    
    def change_voice(self, voice_name):
        """Change the TTS voice"""
        if voice_name in self.voices:
            self.current_voice = voice_name
            print(f"🎭 Voice changed to: {voice_name}")
            # Test the new voice
            self.speak(f"Hello! This is the {voice_name.replace('_', ' ')} voice.")
        else:
            print(f"❌ Voice '{voice_name}' not found. Available: {', '.join(self.voices.keys())}")
    
    def listen(self, timeout=20):
        """Listen for user speech with VAD and convert to text"""
        try:
            with self.microphone as source:
                print("\n🎤 Listening... (speak clearly, pause when done)")
                audio = self.recognizer.listen(source, timeout=timeout, phrase_time_limit=20)
            
            print("🔄 Recognizing speech...")
            text = self.recognizer.recognize_google(audio, language='en-US')
            print(f"✅ You said: \"{text}\"")
            return text
            
        except sr.WaitTimeoutError:
            print("⏱️ No speech detected - timeout")
            return None
        except sr.UnknownValueError:
            print("❌ Could not understand audio - please speak louder and clearer")
            return None
        except sr.RequestError as e:
            print(f"❌ Speech recognition service error: {e}")
            return None
        except Exception as e:
            print(f"❌ Unexpected error: {type(e).__name__}: {e}")
            return None
    
    def speak(self, text):
        """Convert text to speech using Cartesia AI"""
        print(f"\n🤖 Agent: {text}")
        
        try:
            # Prepare Cartesia API request
            headers = {
                "X-API-Key": self.cartesia_key,
                "Cartesia-Version": "2024-06-10",
                "Content-Type": "application/json"
            }
            
            payload = {
                "model_id": "sonic-english",
                "transcript": text,
                "voice": {
                    "mode": "id",
                    "id": self.voices[self.current_voice]
                },
                "output_format": {
                    "container": "mp3",
                    "encoding": "mp3",
                    "sample_rate": 44100
                }
            }
            
            # Make API request
            print("🎭 Generating realistic voice...")
            response = requests.post(
                self.cartesia_url,
                headers=headers,
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                # Load and play audio
                audio_data = BytesIO(response.content)
                pygame.mixer.music.load(audio_data)
                pygame.mixer.music.play()
                
                # Wait for audio to finish
                while pygame.mixer.music.get_busy():
                    pygame.time.Clock().tick(10)
                
                print("🔊 Audio playback complete")
            else:
                print(f"⚠️ Cartesia API error: {response.status_code}")
                print(f"Response: {response.text}")
                print("💡 Falling back to text display")
                
        except requests.exceptions.Timeout:
            print("⚠️ Cartesia API timeout - speech may be too long")
        except Exception as e:
            print(f"⚠️ TTS error: {type(e).__name__}: {e}")
            print("💡 Text displayed above, but voice output failed")
    
    def voice_conversation(self, turns=5):
        """Have a voice conversation with ultra-realistic voice"""
        print("\n" + "="*70)
        print("🎭 ENHANCED VOICE CONVERSATION (Cartesia AI)")
        print("="*70)
        print("💡 Tips:")
        print("   - Speak clearly into your microphone")
        print("   - Pause for 1 second when you finish speaking")
        print("   - Say 'goodbye' to end early")
        print(f"   - Current voice: {self.current_voice}")
        print("="*70)
        
        # Greeting with ultra-realistic voice
        greeting = "Hello! I'm your AI sales assistant. How can I help you today?"
        self.speak(greeting)
        
        successful_turns = 0
        attempts = 0
        max_attempts = turns * 3
        
        # Conversation loop
        while successful_turns < turns and attempts < max_attempts:
            attempts += 1
            print(f"\n{'='*70}")
            print(f"💬 Turn {successful_turns + 1}/{turns} (Attempt {attempts})")
            print(f"{'='*70}")
            
            # Listen to user
            try:
                user_input = self.listen(timeout=25)
            except Exception as e:
                print(f"❌ Error during listening: {type(e).__name__}: {e}")
                user_input = None
            
            if user_input is None:
                if attempts % 2 == 0:
                    self.speak("I didn't catch that. Could you please repeat?")
                else:
                    print("⏭️ Skipping to next attempt...")
                continue
            
            # Check for exit commands
            exit_words = ['goodbye', 'bye', 'exit', 'quit', 'stop', 'end']
            if any(word in user_input.lower() for word in exit_words):
                farewell = "Thank you so much for your time! Have a wonderful day!"
                self.speak(farewell)
                break
            
            # Get AI response
            try:
                print("🤔 Agent is thinking...")
                response = self.chat(user_input)
                self.speak(response)
                successful_turns += 1
            except Exception as e:
                print(f"❌ Error during chat: {type(e).__name__}: {e}")
                error_msg = "I'm having a small technical issue. Let me try that again."
                self.speak(error_msg)
                continue
        
        print("\n" + "="*70)
        print(f"✅ ENHANCED CONVERSATION ENDED ({successful_turns} successful turns)")
        print("="*70)

# Initialize enhanced agent with Cartesia
try:
    print("🎭 Initializing enhanced voice agent with Cartesia TTS...")
    cartesia_agent = CartesiaVoiceAgent(
        api_key=CEREBRAS_API_KEY,
        cartesia_key=CARTESIA_API_KEY,
        use_cerebras=True
    )
    
    print("\n" + "="*70)
    print("📋 CARTESIA VOICE AGENT - USAGE GUIDE")
    print("="*70)
    print("Start conversation with realistic voice:")
    print("  cartesia_agent.voice_conversation(turns=5)")
    print("\nChange voice:")
    print("  cartesia_agent.change_voice('friendly_male')")
    print("  cartesia_agent.change_voice('energetic_female')")
    print("\nTest components:")
    print("  cartesia_agent.listen()  # Test microphone")
    print("  cartesia_agent.speak('Hello world')  # Test Cartesia voice")
    print("="*70)
    
except Exception as e:
    print(f"\n❌ Failed to initialize Cartesia agent: {type(e).__name__}: {e}")
    print("\n💡 Make sure pygame is installed: !pip install pygame")
    import traceback
    print(traceback.format_exc())

pygame 2.6.1 (SDL 2.28.4, Python 3.11.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
🎭 Initializing enhanced voice agent with Cartesia TTS...
🧠 Using Cerebras API (Llama 3.1 8B)
✅ Sales Agent initialized with 1205 characters of product context
✅ Pygame audio initialized
🎭 Using Cartesia TTS - Ultra-realistic voice cloning
🎤 Current voice: professional_female

🎤 Calibrating microphone (please be quiet for 3 seconds)...
✅ Calibration complete! Energy threshold: 65.21163501061294

✅ Enhanced voice agent ready!

⚙️ Available voices: professional_female, friendly_male, energetic_female, calm_male
💡 Change voice: agent.change_voice('voice_name')

📋 CARTESIA VOICE AGENT - USAGE GUIDE
Start conversation with realistic voice:
  cartesia_agent.voice_conversation(turns=5)

Change voice:
  cartesia_agent.change_voice('friendly_male')
  cartesia_agent.change_voice('energetic_female')

Test components:
  cartesia_agent.listen()  # Test microphone
  cartesia_agent.speak('H

In [14]:
# Voice-Enabled Sales Agent with Speech Recognition, TTS, and Improved VAD
import speech_recognition as sr
import pyttsx3
import threading

class VoiceSalesAgent(SalesAgentWithContext):
    def __init__(self, api_key, use_cerebras=True):
        """Initialize voice-enabled sales agent with speech recognition, TTS, and VAD"""
        super().__init__(api_key, use_cerebras)
        
        # Initialize speech recognizer
        self.recognizer = sr.Recognizer()
        self.microphone = sr.Microphone()
        
        # Configure more lenient VAD (Voice Activity Detection) settings
        self.recognizer.energy_threshold = 1000  # Higher = less sensitive to background noise
        self.recognizer.dynamic_energy_threshold = True  # Automatically adjust
        self.recognizer.dynamic_energy_adjustment_damping = 0.15
        self.recognizer.dynamic_energy_ratio = 1.5
        self.recognizer.pause_threshold = 1.0  # 1 second pause before stopping
        self.recognizer.phrase_threshold = 0.3
        self.recognizer.non_speaking_duration = 0.8  # Longer silence tolerance
        
        # Initialize text-to-speech engine with error handling
        print("🔊 Initializing text-to-speech engine...")
        print("✅ Using Windows native TTS (via PowerShell)")
        print("💡 This is more reliable than pyttsx3 on Windows")
        
        # Calibrate for ambient noise
        print("\n🎤 Calibrating microphone (please be quiet for 3 seconds)...")
        with self.microphone as source:
            self.recognizer.adjust_for_ambient_noise(source, duration=3)
            print(f"✅ Calibration complete! Energy threshold: {self.recognizer.energy_threshold}")
        
        print("\n✅ Voice agent ready!")
        print("🎙️ Microphone configured")
        print(f"\n⚙️ VAD Settings:")
        print(f"   - Energy threshold: {self.recognizer.energy_threshold}")
        print(f"   - Pause threshold: {self.recognizer.pause_threshold}s")
        print(f"\n💡 Tip: Speak clearly and pause briefly when done speaking")
    
    def adjust_sensitivity(self, energy_threshold=None):
        """Manually adjust microphone sensitivity"""
        if energy_threshold:
            self.recognizer.energy_threshold = energy_threshold
            print(f"🎚️ Energy threshold adjusted to: {energy_threshold}")
        else:
            with self.microphone as source:
                print("🎤 Re-calibrating...")
                self.recognizer.adjust_for_ambient_noise(source, duration=2)
                print(f"✅ New energy threshold: {self.recognizer.energy_threshold}")
    
    def listen(self, timeout=20):
        """Listen for user speech with VAD and convert to text"""
        try:
            with self.microphone as source:
                print("\n🎤 Listening... (speak clearly, pause when done)")
                
                # Listen for audio
                audio = self.recognizer.listen(
                    source, 
                    timeout=timeout,
                    phrase_time_limit=20
                )
                
            print("🔄 Recognizing speech...")
            
            # Use Google Speech Recognition
            text = self.recognizer.recognize_google(audio, language='en-US')
            print(f"✅ You said: \"{text}\"")
            return text
            
        except sr.WaitTimeoutError:
            print("⏱️ No speech detected - timeout")
            return None
        except sr.UnknownValueError:
            print("❌ Could not understand audio - please speak louder and clearer")
            return None
        except sr.RequestError as e:
            print(f"❌ Speech recognition service error: {e}")
            return None
        except Exception as e:
            print(f"❌ Unexpected error in listen(): {type(e).__name__}: {e}")
            return None
    
    def _convert_numbers_to_words(self, text):
        """Convert numbers in text to words for better TTS pronunciation"""
        import re
        
        # Simple number to words mapping
        ones = ['', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
        teens = ['ten', 'eleven', 'twelve', 'thirteen', 'fourteen', 'fifteen', 
                 'sixteen', 'seventeen', 'eighteen', 'nineteen']
        tens = ['', '', 'twenty', 'thirty', 'forty', 'fifty', 'sixty', 'seventy', 'eighty', 'ninety']
        
        def num_to_words(num):
            """Convert a number (0-9999) to words"""
            try:
                num = int(num)
                if num == 0:
                    return 'zero'
                elif num < 10:
                    return ones[num]
                elif num < 20:
                    return teens[num - 10]
                elif num < 100:
                    tens_digit = num // 10
                    ones_digit = num % 10
                    return tens[tens_digit] + ('' if ones_digit == 0 else ' ' + ones[ones_digit])
                elif num < 1000:
                    hundreds_digit = num // 100
                    remainder = num % 100
                    result = ones[hundreds_digit] + ' hundred'
                    if remainder > 0:
                        result += ' and ' + num_to_words(remainder)
                    return result
                elif num < 1000000:
                    thousands = num // 1000
                    remainder = num % 1000
                    result = num_to_words(thousands) + ' thousand'
                    if remainder > 0:
                        result += ' ' + num_to_words(remainder)
                    return result
                else:
                    return str(num)  # Keep as-is if too large
            except:
                return str(num)
        
        # Find all numbers in text and convert them
        def replace_number(match):
            return num_to_words(match.group(0))
        
        # Replace standalone numbers (not part of words)
        text = re.sub(r'\b\d+\b', replace_number, text)
        return text
    
    def speak(self, text):
        """Convert text to speech and play it using Windows native TTS"""
        print(f"\n🤖 Agent: {text}")
        
        try:
            import subprocess
            import re
            
            # Preprocess text to help Windows TTS pronounce numbers better
            # Add spaces around $ signs so "299" in "$299" is spoken
            text = re.sub(r'\$(\d+)', r'\1 dollars', text)
            text = text.replace('$', ' dollars ')
            
            # Escape quotes in text for PowerShell
            safe_text = text.replace('"', '`"').replace("'", "''")
            
            # Use Windows native TTS via PowerShell with proper synchronous waiting
            ps_command = f'''
            Add-Type -AssemblyName System.Speech;
            $speak = New-Object System.Speech.Synthesis.SpeechSynthesizer;
            $speak.Rate = 1;
            $speak.Volume = 100;
            $speak.Speak("{safe_text}") | Out-Null;
            Start-Sleep -Milliseconds 200;
            '''
            
            # Run PowerShell command and wait for it to complete
            result = subprocess.run(
                ['powershell', '-NoProfile', '-Command', ps_command],
                capture_output=True,
                text=True,
                timeout=60,
                creationflags=subprocess.CREATE_NO_WINDOW if hasattr(subprocess, 'CREATE_NO_WINDOW') else 0
            )
            
            if result.returncode == 0:
                print("🔊 Audio playback complete")
            else:
                print(f"⚠️ TTS error: {result.stderr}")
                print("💡 Text displayed above, but voice output may have failed")
            
        except subprocess.TimeoutExpired:
            print("⚠️ TTS timeout - speech may have been too long")
        except Exception as e:
            print(f"⚠️ TTS error: {type(e).__name__}: {e}")
            print("💡 Text displayed above, but voice output failed")
    
    def voice_conversation(self, turns=5):
        """Have a voice conversation with the agent using VAD"""
        print("\n" + "="*70)
        print("🎙️ VOICE CONVERSATION STARTED")
        print("="*70)
        print("💡 Tips:")
        print("   - Speak clearly into your microphone")
        print("   - Pause for 1 second when you finish speaking")
        print("   - Say 'goodbye' to end early")
        print("="*70)
        
        # Greeting
        greeting = "Hello! I'm your AI sales assistant. How can I help you today?"
        self.speak(greeting)
        
        successful_turns = 0
        attempts = 0
        max_attempts = turns * 3  # Allow retries
        
        # Conversation loop
        while successful_turns < turns and attempts < max_attempts:
            attempts += 1
            print(f"\n{'='*70}")
            print(f"💬 Turn {successful_turns + 1}/{turns} (Attempt {attempts})")
            print(f"{'='*70}")
            
            # Listen to user
            try:
                user_input = self.listen(timeout=25)
            except Exception as e:
                print(f"❌ Error during listening: {type(e).__name__}: {e}")
                user_input = None
            
            if user_input is None:
                if attempts % 2 == 0:
                    self.speak("I didn't catch that. Please try again.")
                else:
                    print("⏭️ Skipping to next attempt...")
                continue
            
            # Check for exit commands
            exit_words = ['goodbye', 'bye', 'exit', 'quit', 'stop', 'end']
            if any(word in user_input.lower() for word in exit_words):
                farewell = "Thank you for your time! Have a great day!"
                self.speak(farewell)
                break
            
            # Get AI response
            try:
                print("🤔 Agent is thinking...")
                response = self.chat(user_input)
                self.speak(response)
                successful_turns += 1  # Only increment on successful turn
            except Exception as e:
                print(f"❌ Error during chat: {type(e).__name__}: {e}")
                import traceback
                print(traceback.format_exc())
                error_msg = "I'm having trouble right now. Let me try that again."
                self.speak(error_msg)
                continue
        
        print("\n" + "="*70)
        print(f"✅ VOICE CONVERSATION ENDED ({successful_turns} successful turns)")
        print("="*70)

# Initialize the voice-enabled agent with improved error handling
try:
    voice_agent = VoiceSalesAgent(api_key=CEREBRAS_API_KEY, use_cerebras=True)
    
    print("\n" + "="*70)
    print("📋 USAGE GUIDE")
    print("="*70)
    print("Start conversation:")
    print("  voice_agent.voice_conversation(turns=5)")
    print("\nTest components:")
    print("  voice_agent.listen()  # Test microphone")
    print("  voice_agent.speak('Hello')  # Test speaker")
    print("\nTroubleshooting:")
    print("  voice_agent.adjust_sensitivity()  # Re-calibrate microphone")
    print("="*70)
    
except Exception as e:
    print(f"\n❌ Failed to initialize voice agent: {type(e).__name__}: {e}")
    import traceback
    print(traceback.format_exc())

🧠 Using Cerebras API (Llama 3.1 8B)
✅ Sales Agent initialized with 1205 characters of product context
🔊 Initializing text-to-speech engine...
✅ Using Windows native TTS (via PowerShell)
💡 This is more reliable than pyttsx3 on Windows

🎤 Calibrating microphone (please be quiet for 3 seconds)...
✅ Calibration complete! Energy threshold: 63.28347530779931

✅ Voice agent ready!
🎙️ Microphone configured

⚙️ VAD Settings:
   - Energy threshold: 63.28347530779931
   - Pause threshold: 1.0s

💡 Tip: Speak clearly and pause briefly when done speaking

📋 USAGE GUIDE
Start conversation:
  voice_agent.voice_conversation(turns=5)

Test components:
  voice_agent.listen()  # Test microphone
  voice_agent.speak('Hello')  # Test speaker

Troubleshooting:
  voice_agent.adjust_sensitivity()  # Re-calibrate microphone


## Cell 8: Enhanced Voice Agent with Cartesia TTS 🎭

**Upgrade to human-like voice quality using Cartesia!**

Features:
- Ultra-realistic voice cloning
- Multiple voice options (male/female, different accents)
- Lower latency than Windows TTS
- Emotional expression in speech

## Cell 9: Install Additional Dependencies (Pygame for Audio)

Required for Cartesia TTS audio playback

In [10]:
# Install pygame for Cartesia audio playback
!pip install pygame requests -q

print("✅ Pygame and requests installed successfully!")

✅ Pygame and requests installed successfully!


In [15]:
# Option 1: Use Cartesia with ultra-realistic voice (RECOMMENDED!)
print("🎭 Starting enhanced conversation with Cartesia AI voice...")
cartesia_agent.voice_conversation(turns=5)

# Want to try a different voice? Uncomment below:
# cartesia_agent.change_voice('friendly_male')
# cartesia_agent.voice_conversation(turns=5)

# Option 2: Compare with original Windows TTS
# voice_agent.voice_conversation(turns=5)

🎭 Starting enhanced conversation with Cartesia AI voice...

🎭 ENHANCED VOICE CONVERSATION (Cartesia AI)
💡 Tips:
   - Speak clearly into your microphone
   - Pause for 1 second when you finish speaking
   - Say 'goodbye' to end early
   - Current voice: professional_female

🤖 Agent: Hello! I'm your AI sales assistant. How can I help you today?
🎭 Generating realistic voice...
🔊 Audio playback complete

💬 Turn 1/5 (Attempt 1)

🎤 Listening... (speak clearly, pause when done)
🔄 Recognizing speech...
✅ You said: "tell me about your most expensive plan"
🤔 Agent is thinking...

🤖 Agent: Our most expensive plan is the Enterprise plan, which costs $299 per month. This plan offers some of our most advanced features, including white-label capabilities, API access, and dedicated support. It's designed for large organizations or businesses that need a high level of customization and support.

Can you tell me a little bit about your business and what you're looking for in a solution?
🎭 Generating rea

# 🏢 Multi-Agent Call Center System

---

## Phase 1: Foundation - Multi-Agent Architecture

Building an intelligent call center with specialized agents:
- **Sales Agent** - Product recommendations, upselling
- **Technical Support** - Troubleshooting, step-by-step guidance  
- **Billing Agent** - Payment issues, account inquiries
- **Appointment Scheduler** - Booking, calendar management

## Cell 11: Base Agent Architecture

In [6]:
# Base Call Center Agent Class
from datetime import datetime
from typing import Dict, List, Optional
import uuid

class CallCenterAgent:
    """Base class for all specialized call center agents"""
    
    def __init__(self, agent_type: str, name: str, personality: str, 
                 api_key: str, use_cerebras: bool = True):
        """
        Initialize a specialized call center agent
        
        Args:
            agent_type: Type of agent (sales, tech_support, billing, appointment)
            name: Agent's name
            personality: Personality description
            api_key: API key for AI service
            use_cerebras: Whether to use Cerebras API
        """
        self.agent_type = agent_type
        self.name = name
        self.personality = personality
        self.api_key = api_key
        self.use_cerebras = use_cerebras
        
        # Initialize AI client
        if use_cerebras:
            self.client = openai.OpenAI(
                api_key=api_key,
                base_url="https://api.cerebras.ai/v1"
            )
            self.model = "llama3.1-8b"
        else:
            self.client = openai.OpenAI(api_key=api_key)
            self.model = "gpt-3.5-turbo"
        
        # Agent capabilities and transfer triggers
        self.can_handle = []  # List of query types this agent can handle
        self.transfer_keywords = []  # Keywords that trigger transfers
        self.conversation_history = []
        
        # Call tracking
        self.call_id = None
        self.call_start_time = None
        self.transfers_received = []
        self.transfers_made = []
        
        print(f"✅ {self.name} ({self.agent_type}) initialized")
    
    def get_system_prompt(self) -> str:
        """Return the specialized system prompt for this agent"""
        raise NotImplementedError("Subclasses must implement get_system_prompt()")
    
    def chat(self, user_message: str) -> str:
        """Process user message and return response"""
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.get_system_prompt()},
                *self.conversation_history
            ],
            temperature=0.7,
            max_tokens=200
        )
        
        assistant_message = response.choices[0].message.content
        self.conversation_history.append({
            "role": "assistant",
            "content": assistant_message
        })
        
        return assistant_message
    
    def should_transfer(self, user_message: str) -> Optional[str]:
        """
        Determine if this query should be transferred to another agent
        Uses AI to intelligently detect when transfer is needed
        
        Returns:
            Target agent type if transfer needed, None otherwise
        """
        # Build list of possible transfer targets based on keywords
        possible_targets = []
        for keyword, target_agent in self.transfer_keywords:
            if target_agent not in possible_targets:
                possible_targets.append(target_agent)
        
        if not possible_targets:
            return None
        
        # Use AI to determine if transfer is needed
        prompt = f"""You are {self.name}, a {self.agent_type} agent.

Customer just said: "{user_message}"

Should this be transferred to another agent?

Available agents:
- sales: Product inquiries, pricing, features, buying
- tech_support: Technical issues, errors, troubleshooting, not working
- billing: Payments, charges, refunds, invoices
- appointment: Scheduling, booking, demos, meetings

Respond with ONLY the agent name to transfer to, or "none" if this query should stay with you.
Examples:
- "my laptop isn't working" → tech_support
- "I want a refund" → billing
- "tell me about this product" → none (stay with current agent if already in sales)"""

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=20
            )
            
            result = response.choices[0].message.content.strip().lower()
            
            # Validate result
            if result in ["sales", "tech_support", "billing", "appointment"] and result != self.agent_type:
                return result
            else:
                return None
                
        except Exception as e:
            print(f"⚠️ Transfer detection error: {e}")
            # Fallback to keyword matching
            user_lower = user_message.lower()
            for keyword, target_agent in self.transfer_keywords:
                if keyword in user_lower:
                    return target_agent
            return None
    
    def start_call(self, call_id: str = None):
        """Start a new call session"""
        self.call_id = call_id or str(uuid.uuid4())[:8]
        self.call_start_time = datetime.now()
        self.conversation_history = []
        print(f"📞 Call {self.call_id} started with {self.name}")
        return self.call_id
    
    def end_call(self):
        """End the current call session"""
        if self.call_start_time:
            duration = (datetime.now() - self.call_start_time).total_seconds()
            print(f"📞 Call {self.call_id} ended. Duration: {duration:.0f}s")
            print(f"   Turns: {len(self.conversation_history) // 2}")
            print(f"   Transfers: {len(self.transfers_made)} out, {len(self.transfers_received)} in")
        self.call_id = None
        self.call_start_time = None
    
    def receive_transfer(self, from_agent: str, context: Dict):
        """Handle incoming transfer from another agent"""
        self.transfers_received.append({
            "from": from_agent,
            "time": datetime.now(),
            "context": context
        })
        print(f"🔄 Transfer received from {from_agent} to {self.name}")
    
    def make_transfer(self, to_agent: str, reason: str):
        """Record outgoing transfer"""
        self.transfers_made.append({
            "to": to_agent,
            "time": datetime.now(),
            "reason": reason
        })
        print(f"🔄 Transferring from {self.name} to {to_agent}: {reason}")
    
    def get_transfer_context(self) -> Dict:
        """Get context to pass during transfer"""
        return {
            "conversation_summary": f"{len(self.conversation_history) // 2} turns",
            "last_user_message": self.conversation_history[-2]["content"] if len(self.conversation_history) >= 2 else "",
            "agent_notes": f"Transferred from {self.name}"
        }

print("✅ Base CallCenterAgent class created")

✅ Base CallCenterAgent class created


## Cell 12: Specialized Agent Classes

Four specialized agents with distinct personalities and expertise

In [9]:
# Specialized Agent Implementations

class SalesAgentSpecialized(CallCenterAgent):
    """Sales specialist - enthusiastic, persuasive, product-focused"""
    
    def __init__(self, api_key: str, use_cerebras: bool = True):
        super().__init__(
            agent_type="sales",
            name="Sarah (Sales)",
            personality="Enthusiastic, persuasive, customer-focused",
            api_key=api_key,
            use_cerebras=use_cerebras
        )
        
        self.can_handle = ["product_inquiry", "pricing", "features", "purchase", "recommendations"]
        self.transfer_keywords = [
            ("not working", "tech_support"),
            ("broken", "tech_support"),
            ("error", "tech_support"),
            ("bug", "tech_support"),
            ("payment", "billing"),
            ("invoice", "billing"),
            ("charge", "billing"),
            ("refund", "billing"),
            ("appointment", "appointment"),
            ("schedule", "appointment"),
            ("book", "appointment"),
        ]
        
        # Load product catalog
        self.context = load_context()
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, an enthusiastic sales specialist.

PERSONALITY: {self.personality}
- Be warm, friendly, and genuinely excited about products
- Focus on benefits and value propositions
- Ask questions to understand customer needs
- Suggest relevant products confidently

AVAILABLE PRODUCTS:
{self.context}

YOUR EXPERTISE:
- Product recommendations based on needs
- Pricing and package comparisons
- Feature explanations and benefits
- Upselling and cross-selling opportunities

COMMUNICATION STYLE:
- Keep responses conversational and concise (2-3 sentences max)
- Sound natural and spoken, not written
- Be enthusiastic but not pushy
- Ask clarifying questions

TRANSFER PROTOCOL:
- If customer mentions technical issues, say: "Let me connect you with our technical team"
- If customer asks about billing/payments, say: "I'll transfer you to our billing specialist"
- If customer wants to schedule something, say: "Let me get you to our scheduling team"

Remember: You're speaking, not writing. Be concise and natural!"""


class TechSupportAgent(CallCenterAgent):
    """Technical support specialist - patient, analytical, step-by-step"""
    
    def __init__(self, api_key: str, use_cerebras: bool = True):
        super().__init__(
            agent_type="tech_support",
            name="Bindu (Tech Support)",
            personality="Patient, analytical, methodical",
            api_key=api_key,
            use_cerebras=use_cerebras
        )
        
        self.can_handle = ["troubleshooting", "technical_issue", "bug", "error", "setup", "configuration"]
        self.transfer_keywords = [
            ("buy", "sales"),
            ("purchase", "sales"),
            ("price", "sales"),
            ("cost", "sales"),
            ("payment", "billing"),
            ("refund", "billing"),
            ("charged", "billing"),
            ("appointment", "appointment"),
        ]
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, a patient technical support specialist.

PERSONALITY: {self.personality}
- Be calm, reassuring, and methodical
- Break down complex problems into simple steps
- Ask diagnostic questions systematically
- Never assume the customer's technical level

YOUR EXPERTISE:
- Software troubleshooting
- Hardware diagnostics
- Setup and configuration
- Error message interpretation
- Step-by-step problem solving

COMMUNICATION STYLE:
- Speak clearly and simply (2-3 sentences)
- One step at a time
- Ask "Have you tried..." questions
- Confirm each step before moving forward
- Be patient and encouraging

TROUBLESHOOTING APPROACH:
1. Understand the problem clearly
2. Ask about recent changes
3. Guide through basic checks first
4. Provide step-by-step solutions
5. Verify the fix worked

TRANSFER PROTOCOL:
- If customer wants to buy/upgrade, say: "Let me connect you with sales"
- If billing issue, say: "I'll transfer you to our billing team"

Remember: Speak naturally. One clear step at a time!"""


class BillingAgent(CallCenterAgent):
    """Billing specialist - precise, empathetic, detail-oriented"""
    
    def __init__(self, api_key: str, use_cerebras: bool = True):
        super().__init__(
            agent_type="billing",
            name="Beth (Billing)",
            personality="Precise, empathetic, detail-oriented",
            api_key=api_key,
            use_cerebras=use_cerebras
        )
        
        self.can_handle = ["payment", "billing", "invoice", "charge", "refund", "subscription", "account"]
        self.transfer_keywords = [
            ("product", "sales"),
            ("buy", "sales"),
            ("features", "sales"),
            ("not working", "tech_support"),
            ("broken", "tech_support"),
            ("error", "tech_support"),
            ("appointment", "appointment"),
        ]
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, a billing and account specialist.

PERSONALITY: {self.personality}
- Be understanding and empathetic about money matters
- Be precise with numbers and details
- Acknowledge customer concerns
- Explain charges clearly

YOUR EXPERTISE:
- Payment processing and issues
- Invoice explanations
- Refund requests and policies
- Subscription management
- Account balance inquiries
- Payment method updates

COMMUNICATION STYLE:
- Be clear and specific about dollar amounts
- Show empathy for billing concerns
- Explain policies without jargon
- Confirm details before processing
- Keep responses brief (2-3 sentences)

BILLING SCENARIOS:
- Unexpected charges: Investigate and explain
- Refund requests: Check eligibility, explain process
- Payment failures: Troubleshoot, suggest alternatives
- Subscription changes: Confirm changes, explain impact

TRANSFER PROTOCOL:
- If customer asks about products, say: "Let me connect you with sales"
- If technical issue, say: "I'll transfer you to technical support"

Remember: Money is sensitive. Be empathetic and precise!"""


class AppointmentAgent(CallCenterAgent):
    """Appointment scheduler - efficient, organized, time-focused"""
    
    def __init__(self, api_key: str, use_cerebras: bool = True):
        super().__init__(
            agent_type="appointment",
            name="Alex (Appointments)",
            personality="Efficient, organized, helpful",
            api_key=api_key,
            use_cerebras=use_cerebras
        )
        
        self.can_handle = ["appointment", "schedule", "booking", "calendar", "meeting", "demo"]
        self.transfer_keywords = [
            ("product", "sales"),
            ("pricing", "sales"),
            ("not working", "tech_support"),
            ("issue", "tech_support"),
            ("payment", "billing"),
            ("charge", "billing"),
        ]
        
        # Simulated available time slots
        self.available_slots = [
            "Monday 10 AM", "Monday 2 PM",
            "Tuesday 9 AM", "Tuesday 3 PM",
            "Wednesday 11 AM", "Wednesday 4 PM",
            "Thursday 1 PM", "Thursday 3 PM",
            "Friday 10 AM", "Friday 2 PM"
        ]
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, an appointment scheduling specialist.

PERSONALITY: {self.personality}
- Be efficient and respectful of time
- Offer specific options quickly
- Confirm details clearly
- Be flexible and helpful

YOUR EXPERTISE:
- Schedule demos and consultations
- Book service appointments
- Reschedule existing appointments
- Send calendar invites
- Manage availability

AVAILABLE TIME SLOTS THIS WEEK:
{', '.join(self.available_slots[:5])}
(and more options available)

COMMUNICATION STYLE:
- Get to the point quickly (2-3 sentences)
- Offer 2-3 specific time options
- Confirm name, time, and type of appointment
- Provide next steps clearly

SCHEDULING PROCESS:
1. Ask what type of appointment (demo, consultation, service)
2. Offer 2-3 time slots
3. Confirm customer's choice
4. Collect contact info (name, email, phone)
5. Confirm booking details

TRANSFER PROTOCOL:
- If customer has questions about products, say: "Let me connect you with sales first"
- If technical issue, say: "Let's get that fixed first with our tech team"

Remember: Time is valuable. Be quick and clear!"""


# Initialize all specialized agents
print("🏢 Creating specialized call center agents...")

sales_agent = SalesAgentSpecialized(api_key=CEREBRAS_API_KEY, use_cerebras=True)
tech_agent = TechSupportAgent(api_key=CEREBRAS_API_KEY, use_cerebras=True)
billing_agent = BillingAgent(api_key=CEREBRAS_API_KEY, use_cerebras=True)
appointment_agent = AppointmentAgent(api_key=CEREBRAS_API_KEY, use_cerebras=True)

print("\n✅ All agents ready!")
print("   - Sarah (Sales)")
print("   - Bindu (Tech Support)")
print("   - Beth (Billing)")
print("   - Alex (Appointments)")

🏢 Creating specialized call center agents...
✅ Sarah (Sales) (sales) initialized
✅ Bindu (Tech Support) (tech_support) initialized
✅ Beth (Billing) (billing) initialized
✅ Alex (Appointments) (appointment) initialized

✅ All agents ready!
   - Sarah (Sales)
   - Bindu (Tech Support)
   - Beth (Billing)
   - Alex (Appointments)


## Cell 13: Intent Classification System

AI-powered router that analyzes caller intent and directs to the right agent

In [10]:
# Intent Classification System

class IntentClassifier:
    """AI-powered intent detection for intelligent call routing"""
    
    def __init__(self, api_key: str, use_cerebras: bool = True):
        """Initialize the intent classifier"""
        if use_cerebras:
            self.client = openai.OpenAI(
                api_key=api_key,
                base_url="https://api.cerebras.ai/v1"
            )
            self.model = "llama3.1-8b"
        else:
            self.client = openai.OpenAI(api_key=api_key)
            self.model = "gpt-3.5-turbo"
        
        self.intent_map = {
            "sales": ["product", "buy", "purchase", "price", "cost", "feature", "recommend", "looking for"],
            "tech_support": ["not working", "broken", "error", "bug", "issue", "problem", "help", "fix"],
            "billing": ["payment", "charge", "invoice", "bill", "refund", "subscription", "account"],
            "appointment": ["schedule", "appointment", "book", "calendar", "demo", "meeting", "time"]
        }
        
        print("✅ Intent Classifier initialized")
    
    def classify(self, user_message: str, use_ai: bool = True) -> tuple:
        """
        Classify user intent and determine appropriate agent
        
        Args:
            user_message: The user's message
            use_ai: Whether to use AI classification (True) or keyword matching (False)
        
        Returns:
            Tuple of (agent_type, confidence_score)
        """
        if use_ai:
            return self._classify_with_ai(user_message)
        else:
            return self._classify_with_keywords(user_message)
    
    def _classify_with_ai(self, user_message: str) -> tuple:
        """Use AI to classify intent"""
        prompt = f"""Classify this customer inquiry into ONE category:

Customer: "{user_message}"

Categories:
- sales: Product inquiries, pricing, features, buying, recommendations
- tech_support: Technical problems, errors, bugs, troubleshooting
- billing: Payments, charges, refunds, invoices, account issues
- appointment: Scheduling, booking, calendar, demos, meetings

Respond with ONLY the category name, nothing else."""

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=20
            )
            
            intent = response.choices[0].message.content.strip().lower()
            
            # Validate intent
            if intent in ["sales", "tech_support", "billing", "appointment"]:
                return (intent, 0.9)  # High confidence for AI classification
            else:
                # Fallback to keyword matching if AI returns invalid intent
                return self._classify_with_keywords(user_message)
                
        except Exception as e:
            print(f"⚠️ AI classification error: {e}")
            return self._classify_with_keywords(user_message)
    
    def _classify_with_keywords(self, user_message: str) -> tuple:
        """Use keyword matching as fallback"""
        user_lower = user_message.lower()
        scores = {agent_type: 0 for agent_type in self.intent_map.keys()}
        
        # Score each intent based on keyword matches
        for agent_type, keywords in self.intent_map.items():
            for keyword in keywords:
                if keyword in user_lower:
                    scores[agent_type] += 1
        
        # Find highest scoring intent
        if max(scores.values()) > 0:
            best_intent = max(scores, key=scores.get)
            confidence = min(scores[best_intent] * 0.3, 0.9)  # Max 0.9 for keyword matching
            return (best_intent, confidence)
        else:
            # Default to sales if no clear intent
            return ("sales", 0.3)
    
    def explain_classification(self, user_message: str, intent: str, confidence: float):
        """Print explanation of classification"""
        print(f"\n🎯 Intent Analysis:")
        print(f"   Message: \"{user_message}\"")
        print(f"   → Classified as: {intent}")
        print(f"   → Confidence: {confidence:.0%}")


# Initialize intent classifier
intent_classifier = IntentClassifier(api_key=CEREBRAS_API_KEY, use_cerebras=True)

# Test the classifier
print("\n🧪 Testing Intent Classifier:")
test_queries = [
    "I want to buy a laptop",
    "My computer is not working",
    "I was charged twice",
    "Can I schedule a demo?"
]

for query in test_queries:
    intent, confidence = intent_classifier.classify(query, use_ai=True)
    intent_classifier.explain_classification(query, intent, confidence)

✅ Intent Classifier initialized

🧪 Testing Intent Classifier:

🎯 Intent Analysis:
   Message: "I want to buy a laptop"
   → Classified as: sales
   → Confidence: 90%

🎯 Intent Analysis:
   Message: "My computer is not working"
   → Classified as: tech_support
   → Confidence: 90%

🎯 Intent Analysis:
   Message: "I was charged twice"
   → Classified as: billing
   → Confidence: 90%

🎯 Intent Analysis:
   Message: "Can I schedule a demo?"
   → Classified as: appointment
   → Confidence: 90%


## Cell 14: Call Center Manager

Routes calls, manages transfers, and tracks all conversations

In [12]:
# Call Center Manager - Routes calls and handles transfers

class CallCenterManager:
    """Manages all agents, routes calls, and handles transfers"""
    
    def __init__(self, intent_classifier: IntentClassifier):
        """Initialize the call center manager"""
        self.classifier = intent_classifier
        
        # Register all agents
        self.agents = {
            "sales": sales_agent,
            "tech_support": tech_agent,
            "billing": billing_agent,
            "appointment": appointment_agent
        }
        
        # Call tracking
        self.active_calls = {}  # call_id -> current_agent_type
        self.call_logs = []
        
        print("✅ Call Center Manager initialized")
        print(f"   Managing {len(self.agents)} specialized agents")
    
    def route_call(self, user_message: str, use_ai_routing: bool = True) -> CallCenterAgent:
        """
        Route a new call to the appropriate agent
        
        Args:
            user_message: Initial user message
            use_ai_routing: Use AI for intent detection (True) or keywords (False)
        
        Returns:
            The appropriate agent to handle the call
        """
        intent, confidence = self.classifier.classify(user_message, use_ai=use_ai_routing)
        agent = self.agents[intent]
        
        print(f"\n📞 Routing call to {agent.name}")
        print(f"   Intent: {intent} (confidence: {confidence:.0%})")
        
        return agent
    
    def handle_transfer(self, current_agent: CallCenterAgent, user_message: str) -> Optional[CallCenterAgent]:
        """
        Check if call should be transferred to different agent
        
        Args:
            current_agent: Currently handling agent
            user_message: Latest user message
        
        Returns:
            New agent if transfer needed, None otherwise
        """
        # Check if current agent wants to transfer
        target_type = current_agent.should_transfer(user_message)
        
        if target_type and target_type in self.agents:
            new_agent = self.agents[target_type]
            
            # Record the transfer
            context = current_agent.get_transfer_context()
            current_agent.make_transfer(new_agent.name, "Customer needs different expertise")
            new_agent.receive_transfer(current_agent.name, context)
            
            return new_agent
        
        return None
    
    def start_call(self, initial_message: str, call_id: str = None) -> tuple:
        """
        Start a new call session
        
        Returns:
            Tuple of (agent, call_id)
        """
        # Route to appropriate agent
        agent = self.route_call(initial_message)
        
        # Start call session
        call_id = agent.start_call(call_id)
        self.active_calls[call_id] = agent.agent_type
        
        return agent, call_id
    
    def end_call(self, agent: CallCenterAgent):
        """End a call session"""
        if agent.call_id in self.active_calls:
            del self.active_calls[agent.call_id]
        
        # Log the call
        self.call_logs.append({
            "call_id": agent.call_id,
            "agent_type": agent.agent_type,
            "agent_name": agent.name,
            "start_time": agent.call_start_time,
            "end_time": datetime.now(),
            "turns": len(agent.conversation_history) // 2,
            "transfers_in": len(agent.transfers_received),
            "transfers_out": len(agent.transfers_made)
        })
        
        agent.end_call()
    
    def get_stats(self):
        """Get call center statistics"""
        if not self.call_logs:
            print("📊 No calls recorded yet")
            return
        
        print("\n📊 Call Center Statistics:")
        print(f"   Total calls: {len(self.call_logs)}")
        print(f"   Active calls: {len(self.active_calls)}")
        
        # Agent breakdown
        agent_calls = {}
        for log in self.call_logs:
            agent_type = log["agent_type"]
            agent_calls[agent_type] = agent_calls.get(agent_type, 0) + 1
        
        print("\n   Calls by agent:")
        for agent_type, count in agent_calls.items():
            agent_name = self.agents[agent_type].name
            print(f"     - {agent_name}: {count}")
        
        # Transfer stats
        total_transfers = sum(log["transfers_out"] for log in self.call_logs)
        print(f"\n   Total transfers: {total_transfers}")
        
        # Average call duration (only for calls with valid timestamps)
        durations = [
            (log["end_time"] - log["start_time"]).total_seconds() 
            for log in self.call_logs 
            if log.get("start_time") and log.get("end_time")
        ]
        avg_duration = sum(durations) / len(durations) if durations else 0
        print(f"   Average call duration: {avg_duration:.0f}s")


# Initialize Call Center Manager
call_center = CallCenterManager(intent_classifier)

print("\n" + "="*70)
print("🏢 CALL CENTER READY")
print("="*70)
print("Available agents:")
print("  - Sarah (Sales) - Product inquiries, recommendations")
print("  - Bindu (Tech Support) - Technical issues, troubleshooting")
print("  - Beth (Billing) - Payments, refunds, account issues")
print("  - Alex (Appointments) - Scheduling, bookings")
print("="*70)

✅ Call Center Manager initialized
   Managing 4 specialized agents

🏢 CALL CENTER READY
Available agents:
  - Sarah (Sales) - Product inquiries, recommendations
  - Bindu (Tech Support) - Technical issues, troubleshooting
  - Beth (Billing) - Payments, refunds, account issues
  - Alex (Appointments) - Scheduling, bookings


## Cell 15: Test Multi-Agent System (Text-Based)

Test the routing and transfer system with text conversations

In [13]:
# Test the Multi-Agent Call Center System

def test_conversation(initial_message: str, follow_ups: List[str]):
    """Test a complete conversation with routing and transfers"""
    print("\n" + "="*70)
    print("🧪 TESTING MULTI-AGENT CONVERSATION")
    print("="*70)
    
    # Start call with initial message
    agent, call_id = call_center.start_call(initial_message)
    
    # Get initial response
    response = agent.chat(initial_message)
    print(f"\n{agent.name}: {response}")
    
    # Handle follow-up messages
    for user_msg in follow_ups:
        print(f"\nCustomer: {user_msg}")
        
        # Check if transfer needed
        new_agent = call_center.handle_transfer(agent, user_msg)
        if new_agent:
            agent = new_agent
            print(f"\n🔄 TRANSFERRED TO {agent.name}\n")
            
            # Warm handoff message
            handoff_msg = f"Hi! I understand you need help. How can I assist you?"
            print(f"{agent.name}: {handoff_msg}")
        
        # Get response from current agent
        response = agent.chat(user_msg)
        print(f"{agent.name}: {response}")
    
    # End call
    call_center.end_call(agent)
    print("\n" + "="*70)


# Test Scenario 1: Sales to Tech Support transfer
print("📝 Scenario 1: Customer starts with product inquiry, then has technical issue")
test_conversation(
    initial_message="I'm interested in buying a laptop",
    follow_ups=[
        "What's the most powerful one you have?",
        "Actually, my current laptop isn't working properly. Can you help?"
    ]
)

# Test Scenario 2: Tech Support to Billing transfer
print("\n\n📝 Scenario 2: Technical issue that leads to refund request")
test_conversation(
    initial_message="My software keeps crashing",
    follow_ups=[
        "I've tried restarting but it still doesn't work",
        "This is frustrating. I want a refund."
    ]
)

# Test Scenario 3: Direct appointment booking
print("\n\n📝 Scenario 3: Direct appointment scheduling")
test_conversation(
    initial_message="I'd like to schedule a product demo",
    follow_ups=[
        "What times do you have available?",
        "Tuesday afternoon works for me"
    ]
)

# Show statistics
print("\n")
call_center.get_stats()

📝 Scenario 1: Customer starts with product inquiry, then has technical issue

🧪 TESTING MULTI-AGENT CONVERSATION

📞 Routing call to Sarah (Sales)
   Intent: sales (confidence: 90%)
📞 Call bb7cb998 started with Sarah (Sales)

Sarah (Sales): So you're in the market for a new laptop, huh? What's the primary use? Are you looking to use it for work, school, or just personal stuff like browsing and streaming?

Customer: What's the most powerful one you have?
Sarah (Sales): We've got some real beasts in our lineup. Our top-of-the-line laptop features an Intel Core i9 processor, 64GB of RAM, and a 2TB solid-state drive. It's a real powerhouse that can handle demanding tasks like video editing and 3D modeling with ease. Would you like to take a look at the specs and pricing?

Customer: Actually, my current laptop isn't working properly. Can you help?
🔄 Transferring from Sarah (Sales) to Bindu (Tech Support): Customer needs different expertise
🔄 Transfer received from Sarah (Sales) to Bindu (Tec

# 🎙️ Phase 2: Voice-Enabled Multi-Agent System

---

## Add Voice Capabilities to Call Center

Integrating:
- **Cartesia TTS** - Ultra-realistic voice for each agent
- **Speech Recognition** - Listen to customer voice input
- **Voice Activity Detection** - Smooth conversation flow
- **Agent-Specific Voices** - Each agent has unique voice personality

## Cell 16: Voice-Enabled Call Center Agent

In [29]:
# Voice-Enabled Call Center Agent with Cartesia TTS
import requests
from io import BytesIO
import pygame
import speech_recognition as sr

class VoiceCallCenterAgent(CallCenterAgent):
    """Enhanced CallCenterAgent with voice capabilities"""
    
    # Voice profiles for each agent type
    VOICE_PROFILES = {
        "sales": {
            "voice_id": "694f9389-aac1-45b6-b726-9d9369183238",  # Professional female
            "name": "Professional Female",
            "rate": 1.0
        },
        "tech_support": {
            "voice_id": "694f9389-aac1-45b6-b726-9d9369183238",  # Calm female (professional)
            "name": "Calm Female",
            "rate": 0.95
        },
        "billing": {
            "voice_id": "f114a467-c40a-4db8-964d-aaba89cd08fa",  # Energetic female
            "name": "Energetic Female",
            "rate": 1.0
        },
        "appointment": {
            "voice_id": "79a125e8-cd45-4c13-8a67-188112f4dd22",  # Friendly male
            "name": "Friendly Male",
            "rate": 1.05
        }
    }
    
    def __init__(self, agent_type: str, name: str, personality: str,
                 api_key: str, cartesia_key: str, use_cerebras: bool = True,
                 enable_voice: bool = True):
        """Initialize voice-enabled agent"""
        super().__init__(agent_type, name, personality, api_key, use_cerebras)
        
        self.enable_voice = enable_voice
        self.cartesia_key = cartesia_key
        self.cartesia_url = "https://api.cartesia.ai/tts/bytes"
        
        # Set voice profile for this agent type
        self.voice_profile = self.VOICE_PROFILES.get(agent_type, self.VOICE_PROFILES["sales"])
        
        if enable_voice:
            # Initialize speech recognizer
            self.recognizer = sr.Recognizer()
            self.microphone = sr.Microphone()
            
            # Configure VAD settings - balanced for natural conversation
            self.recognizer.energy_threshold = 1000
            self.recognizer.dynamic_energy_threshold = True
            self.recognizer.pause_threshold = 1.2  # Wait 1.2 seconds of silence (good balance)
            self.recognizer.phrase_threshold = 0.4  # Minimum 0.4s of speech required
            self.recognizer.non_speaking_duration = 1.0  # Moderate audio context
            
            # Initialize pygame for audio playback
            if not pygame.mixer.get_init():
                pygame.mixer.init()
            
            # Calibrate microphone
            print(f"🎤 Calibrating microphone for {self.name}...")
            with self.microphone as source:
                self.recognizer.adjust_for_ambient_noise(source, duration=2)
            
            print(f"✅ {self.name} voice-enabled with {self.voice_profile['name']}")
    
    def listen(self, timeout: int = 20, prompt: str = None) -> Optional[str]:
        """Listen for user speech and convert to text"""
        if not self.enable_voice:
            return None
        
        try:
            if prompt:
                print(f"\n💬 {prompt}")
            
            with self.microphone as source:
                print("🎤 Listening... (speak clearly, pause when done)")
                audio = self.recognizer.listen(source, timeout=timeout, phrase_time_limit=20)
            
            print("🔄 Recognizing speech...")
            text = self.recognizer.recognize_google(audio, language='en-US')
            print(f"✅ Customer: \"{text}\"")
            return text
            
        except sr.WaitTimeoutError:
            print("⏱️ No speech detected - timeout")
            return None
        except sr.UnknownValueError:
            print("❌ Could not understand audio")
            return None
        except sr.RequestError as e:
            print(f"❌ Speech recognition error: {e}")
            return None
        except Exception as e:
            print(f"❌ Unexpected error: {e}")
            return None
    
    def speak(self, text: str, override_voice: bool = False):
        """Convert text to speech using Cartesia with agent-specific voice"""
        print(f"\n🤖 {self.name}: {text}")
        
        if not self.enable_voice:
            return
        
        try:
            headers = {
                "X-API-Key": self.cartesia_key,
                "Cartesia-Version": "2024-06-10",
                "Content-Type": "application/json"
            }
            
            payload = {
                "model_id": "sonic-english",
                "transcript": text,
                "voice": {
                    "mode": "id",
                    "id": self.voice_profile["voice_id"]
                },
                "output_format": {
                    "container": "mp3",
                    "encoding": "mp3",
                    "sample_rate": 44100
                }
            }
            
            response = requests.post(
                self.cartesia_url,
                headers=headers,
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                audio_data = BytesIO(response.content)
                pygame.mixer.music.load(audio_data)
                pygame.mixer.music.play()
                
                while pygame.mixer.music.get_busy():
                    pygame.time.Clock().tick(10)
                
                print("🔊 Speech complete")
            else:
                print(f"⚠️ TTS error: {response.status_code}")
                
        except Exception as e:
            print(f"⚠️ TTS error: {e}")
    
    def voice_chat(self, user_message: str) -> str:
        """Process message and speak response"""
        response = self.chat(user_message)
        self.speak(response)
        return response
    
    def voice_conversation_turn(self, timeout: int = 25) -> Optional[tuple]:
        """
        One turn of voice conversation: listen, process, respond
        
        Returns:
            Tuple of (user_message, agent_response) or None if no input
        """
        user_message = self.listen(timeout=timeout)
        
        if user_message:
            response = self.voice_chat(user_message)
            return (user_message, response)
        
        return None


print("✅ VoiceCallCenterAgent class created")
print("🎭 Voice profiles configured:")
for agent_type, profile in VoiceCallCenterAgent.VOICE_PROFILES.items():
    print(f"   - {agent_type}: {profile['name']}")

✅ VoiceCallCenterAgent class created
🎭 Voice profiles configured:
   - sales: Professional Female
   - tech_support: Calm Female
   - billing: Energetic Female
   - appointment: Friendly Male


## Cell 17: Voice-Enabled Specialized Agents

Create voice versions of each agent with unique voices

In [30]:
# Voice-Enabled Specialized Agents

class VoiceSalesAgent(VoiceCallCenterAgent):
    """Voice-enabled Sales Agent"""
    
    def __init__(self, api_key: str, cartesia_key: str, use_cerebras: bool = True, enable_voice: bool = True):
        super().__init__(
            agent_type="sales",
            name="Sarah (Sales)",
            personality="Enthusiastic, persuasive, customer-focused",
            api_key=api_key,
            cartesia_key=cartesia_key,
            use_cerebras=use_cerebras,
            enable_voice=enable_voice
        )
        
        self.can_handle = ["product_inquiry", "pricing", "features", "purchase", "recommendations"]
        self.transfer_keywords = [
            ("not working", "tech_support"),
            ("broken", "tech_support"),
            ("error", "tech_support"),
            ("bug", "tech_support"),
            ("payment", "billing"),
            ("invoice", "billing"),
            ("charge", "billing"),
            ("refund", "billing"),
            ("appointment", "appointment"),
            ("schedule", "appointment"),
            ("book", "appointment"),
        ]
        
        self.context = load_context()
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, an enthusiastic sales specialist.

PERSONALITY: {self.personality}
- Be warm, friendly, and genuinely excited about products
- Focus on benefits and value propositions
- Ask questions to understand customer needs
- Suggest relevant products confidently

AVAILABLE PRODUCTS:
{self.context}

COMMUNICATION STYLE:
- Keep responses conversational and concise (2-3 sentences max)
- Sound natural and spoken, not written
- Be enthusiastic but not pushy

TRANSFER PROTOCOL:
- If customer mentions technical issues, say: "Let me connect you with our technical team"
- If customer asks about billing/payments, say: "I'll transfer you to our billing specialist"
- If customer wants to schedule something, say: "Let me get you to our scheduling team"

Remember: You're speaking, not writing. Be concise and natural!"""


class VoiceTechSupportAgent(VoiceCallCenterAgent):
    """Voice-enabled Tech Support Agent"""
    
    def __init__(self, api_key: str, cartesia_key: str, use_cerebras: bool = True, enable_voice: bool = True):
        super().__init__(
            agent_type="tech_support",
            name="Bindu (Tech Support)",
            personality="Patient, analytical, methodical",
            api_key=api_key,
            cartesia_key=cartesia_key,
            use_cerebras=use_cerebras,
            enable_voice=enable_voice
        )
        
        self.can_handle = ["troubleshooting", "technical_issue", "bug", "error", "setup", "configuration"]
        self.transfer_keywords = [
            ("buy", "sales"),
            ("purchase", "sales"),
            ("price", "sales"),
            ("cost", "sales"),
            ("payment", "billing"),
            ("refund", "billing"),
            ("charged", "billing"),
            ("appointment", "appointment"),
        ]
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, a patient technical support specialist.

PERSONALITY: {self.personality}
- Be calm, reassuring, and methodical
- Break down complex problems into simple steps
- Ask diagnostic questions systematically

COMMUNICATION STYLE:
- Speak clearly and simply (2-3 sentences)
- One step at a time
- Be patient and encouraging

TROUBLESHOOTING APPROACH:
1. Understand the problem clearly
2. Ask about recent changes
3. Guide through basic checks first
4. Provide step-by-step solutions

TRANSFER PROTOCOL:
- If customer wants to buy/upgrade, say: "Let me connect you with sales"
- If billing issue, say: "I'll transfer you to our billing team"

Remember: Speak naturally. One clear step at a time!"""


class VoiceBillingAgent(VoiceCallCenterAgent):
    """Voice-enabled Billing Agent"""
    
    def __init__(self, api_key: str, cartesia_key: str, use_cerebras: bool = True, enable_voice: bool = True):
        super().__init__(
            agent_type="billing",
            name="Beth (Billing)",
            personality="Precise, empathetic, detail-oriented",
            api_key=api_key,
            cartesia_key=cartesia_key,
            use_cerebras=use_cerebras,
            enable_voice=enable_voice
        )
        
        self.can_handle = ["payment", "billing", "invoice", "charge", "refund", "subscription", "account"]
        self.transfer_keywords = [
            ("product", "sales"),
            ("buy", "sales"),
            ("features", "sales"),
            ("not working", "tech_support"),
            ("broken", "tech_support"),
            ("error", "tech_support"),
            ("appointment", "appointment"),
        ]
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, a billing and account specialist.

PERSONALITY: {self.personality}
- Be understanding and empathetic about money matters
- Be precise with numbers and details
- Acknowledge customer concerns

COMMUNICATION STYLE:
- Be clear and specific about dollar amounts
- Show empathy for billing concerns
- Keep responses brief (2-3 sentences)

TRANSFER PROTOCOL:
- If customer asks about products, say: "Let me connect you with sales"
- If technical issue, say: "I'll transfer you to technical support"

Remember: Money is sensitive. Be empathetic and precise!"""


class VoiceAppointmentAgent(VoiceCallCenterAgent):
    """Voice-enabled Appointment Agent"""
    
    def __init__(self, api_key: str, cartesia_key: str, use_cerebras: bool = True, enable_voice: bool = True):
        super().__init__(
            agent_type="appointment",
            name="Alex (Appointments)",
            personality="Efficient, organized, helpful",
            api_key=api_key,
            cartesia_key=cartesia_key,
            use_cerebras=use_cerebras,
            enable_voice=enable_voice
        )
        
        self.can_handle = ["appointment", "schedule", "booking", "calendar", "meeting", "demo"]
        self.transfer_keywords = [
            ("product", "sales"),
            ("pricing", "sales"),
            ("not working", "tech_support"),
            ("issue", "tech_support"),
            ("payment", "billing"),
            ("charge", "billing"),
        ]
        
        self.available_slots = [
            "Monday 10 AM", "Monday 2 PM",
            "Tuesday 9 AM", "Tuesday 3 PM",
            "Wednesday 11 AM", "Wednesday 4 PM",
            "Thursday 1 PM", "Thursday 3 PM",
            "Friday 10 AM", "Friday 2 PM"
        ]
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, an appointment scheduling specialist.

PERSONALITY: {self.personality}
- Be efficient and respectful of time
- Offer specific options quickly
- Be flexible and helpful

AVAILABLE TIME SLOTS THIS WEEK:
{', '.join(self.available_slots[:5])}

COMMUNICATION STYLE:
- Get to the point quickly (2-3 sentences)
- Offer 2-3 specific time options
- Confirm details clearly

TRANSFER PROTOCOL:
- If customer has questions about products, say: "Let me connect you with sales first"
- If technical issue, say: "Let's get that fixed first with our tech team"

Remember: Time is valuable. Be quick and clear!"""


# Initialize voice-enabled agents
print("🎙️ Creating voice-enabled agents...")

voice_sales = VoiceSalesAgent(
    api_key=CEREBRAS_API_KEY,
    cartesia_key=CARTESIA_API_KEY,
    use_cerebras=True,
    enable_voice=True
)

voice_tech = VoiceTechSupportAgent(
    api_key=CEREBRAS_API_KEY,
    cartesia_key=CARTESIA_API_KEY,
    use_cerebras=True,
    enable_voice=True
)

voice_billing = VoiceBillingAgent(
    api_key=CEREBRAS_API_KEY,
    cartesia_key=CARTESIA_API_KEY,
    use_cerebras=True,
    enable_voice=True
)

voice_appointment = VoiceAppointmentAgent(
    api_key=CEREBRAS_API_KEY,
    cartesia_key=CARTESIA_API_KEY,
    use_cerebras=True,
    enable_voice=True
)

print("\n✅ All voice agents ready!")
print("   🎤 Sarah (Sales) - Professional Female voice")
print("   🎤 Bindu (Tech Support) - Calm Female voice")
print("   🎤 Beth (Billing) - Energetic Female voice")
print("   🎤 Alex (Appointments) - Friendly Male voice")

🎙️ Creating voice-enabled agents...
✅ Sarah (Sales) (sales) initialized
🎤 Calibrating microphone for Sarah (Sales)...
✅ Sarah (Sales) voice-enabled with Professional Female
✅ Bindu (Tech Support) (tech_support) initialized
🎤 Calibrating microphone for Bindu (Tech Support)...
✅ Bindu (Tech Support) voice-enabled with Calm Female
✅ Beth (Billing) (billing) initialized
🎤 Calibrating microphone for Beth (Billing)...
✅ Beth (Billing) voice-enabled with Energetic Female
✅ Alex (Appointments) (appointment) initialized
🎤 Calibrating microphone for Alex (Appointments)...
✅ Alex (Appointments) voice-enabled with Friendly Male

✅ All voice agents ready!
   🎤 Sarah (Sales) - Professional Female voice
   🎤 Bindu (Tech Support) - Calm Female voice
   🎤 Beth (Billing) - Energetic Female voice
   🎤 Alex (Appointments) - Friendly Male voice


## Cell 18: Voice Call Center Manager

Manages voice conversations with routing and transfers

In [31]:
# Voice-Enabled Call Center Manager

class VoiceCallCenterManager(CallCenterManager):
    """Enhanced manager for voice-enabled agents"""
    
    def __init__(self, intent_classifier: IntentClassifier, voice_agents: dict):
        """Initialize with voice-enabled agents"""
        self.classifier = intent_classifier
        self.agents = voice_agents
        self.active_calls = {}
        self.call_logs = []
        
        print("✅ Voice Call Center Manager initialized")
        print(f"   Managing {len(self.agents)} voice-enabled agents")
    
    def voice_conversation(self, max_turns: int = 10):
        """
        Conduct a full voice conversation with routing and transfers
        
        Args:
            max_turns: Maximum number of conversation turns
        """
        print("\n" + "="*70)
        print("🎙️ VOICE CALL CENTER - STARTING CONVERSATION")
        print("="*70)
        print("💡 Speak clearly into your microphone")
        print("💡 The AI will route you to the right agent")
        print("💡 Transfers happen automatically when needed")
        print("💡 Say 'goodbye' to end the call")
        print("="*70)
        
        # Initial greeting and routing
        greeting_agent = self.agents["sales"]  # Default greeter
        greeting_agent.speak("Hello! Thank you for calling. How can I help you today?")
        
        # Listen to initial request
        initial_message = greeting_agent.listen(timeout=25)
        
        if not initial_message:
            greeting_agent.speak("I didn't catch that. Please call back when you're ready.")
            return
        
        # Route to appropriate agent
        agent, call_id = self.start_call(initial_message)
        
        # Warm handoff if different from greeter
        if agent.agent_type != "sales":
            handoff_msg = f"Let me connect you with {agent.name.split('(')[0].strip()}."
            greeting_agent.speak(handoff_msg)
            agent.speak(f"Hi! I'm {agent.name.split('(')[0].strip()}. I understand you need help. Let me assist you.")
        
        # Get initial response
        response = agent.voice_chat(initial_message)
        
        # Conversation loop
        turn_count = 1
        while turn_count < max_turns:
            print(f"\n{'='*70}")
            print(f"💬 Turn {turn_count + 1}/{max_turns}")
            print(f"{'='*70}")
            
            # Listen to customer
            user_message = agent.listen(timeout=30)
            
            if not user_message:
                agent.speak("Are you still there? I'll wait a moment.")
                user_message = agent.listen(timeout=15)
                
                if not user_message:
                    agent.speak("I haven't heard anything. Feel free to call back anytime. Goodbye!")
                    break
            
            # Check for goodbye
            exit_words = ['goodbye', 'bye', 'thank you bye', 'that\'s all', 'end call']
            if any(word in user_message.lower() for word in exit_words):
                agent.speak("Thank you for calling! Have a wonderful day!")
                break
            
            # Check if transfer needed
            new_agent = self.handle_transfer(agent, user_message)
            
            if new_agent:
                # Warm handoff
                handoff_msg = f"I'm transferring you to {new_agent.name.split('(')[0].strip()} who can better help with this."
                agent.speak(handoff_msg)
                
                print(f"\n🔄 TRANSFERRED TO {new_agent.name}\n")
                
                agent = new_agent
                agent.speak(f"Hi! I'm {agent.name.split('(')[0].strip()}. I see you need assistance. How can I help?")
            
            # Get and speak response
            response = agent.voice_chat(user_message)
            turn_count += 1
        
        # End call
        self.end_call(agent)
        
        print("\n" + "="*70)
        print("📞 CALL ENDED")
        print("="*70)
        self.get_stats()


# Initialize Voice Call Center
voice_call_center = VoiceCallCenterManager(
    intent_classifier=intent_classifier,
    voice_agents={
        "sales": voice_sales,
        "tech_support": voice_tech,
        "billing": voice_billing,
        "appointment": voice_appointment
    }
)

print("\n" + "="*70)
print("🎙️ VOICE CALL CENTER READY")
print("="*70)
print("Voice-enabled agents:")
print("  🎤 Sarah (Sales) - Professional Female")
print("  🎤 Bindu (Tech Support) - Calm Female")
print("  🎤 Beth (Billing) - Energetic Female")
print("  🎤 Alex (Appointments) - Friendly Male")
print("\nTo start:")
print("  voice_call_center.voice_conversation(max_turns=10)")
print("="*70)

✅ Voice Call Center Manager initialized
   Managing 4 voice-enabled agents

🎙️ VOICE CALL CENTER READY
Voice-enabled agents:
  🎤 Sarah (Sales) - Professional Female
  🎤 Bindu (Tech Support) - Calm Female
  🎤 Beth (Billing) - Energetic Female
  🎤 Alex (Appointments) - Friendly Male

To start:
  voice_call_center.voice_conversation(max_turns=10)


## Cell 19: Test Voice Call Center

Start a voice conversation - speak naturally and the system will route and transfer automatically!

In [32]:
# Start a voice conversation with the call center
# Uncomment to test:

# voice_call_center.voice_conversation(max_turns=10)

print("✅ Ready to start voice conversation!")
print("\n🎙️ To begin, run:")
print("   voice_call_center.voice_conversation(max_turns=10)")
print("\n💡 Tips:")
print("   - Make sure your microphone is working")
print("   - Speak clearly and naturally")
print("   - The system will automatically route you to the right agent")
print("   - Transfers happen seamlessly when needed")
print("   - Say 'goodbye' to end the call")
print("\n📝 Example scenarios to try:")
print("   - 'I want to buy a laptop'")
print("   - 'My software isn't working'")
print("   - 'I need a refund'")
print("   - 'I'd like to schedule a demo'")

✅ Ready to start voice conversation!

🎙️ To begin, run:
   voice_call_center.voice_conversation(max_turns=10)

💡 Tips:
   - Make sure your microphone is working
   - Speak clearly and naturally
   - The system will automatically route you to the right agent
   - Transfers happen seamlessly when needed
   - Say 'goodbye' to end the call

📝 Example scenarios to try:
   - 'I want to buy a laptop'
   - 'My software isn't working'
   - 'I need a refund'
   - 'I'd like to schedule a demo'


In [ ]:
# Start the voice call center conversation
voice_call_center.voice_conversation(max_turns=10)


🎙️ VOICE CALL CENTER - STARTING CONVERSATION
💡 Speak clearly into your microphone
💡 The AI will route you to the right agent
💡 Transfers happen automatically when needed
💡 Say 'goodbye' to end the call

🤖 Sarah (Sales): Hello! Thank you for calling. How can I help you today?
🔊 Speech complete
🎤 Listening... (speak clearly, pause when done)
🔄 Recognizing speech...
✅ Customer: "I would like to book an appointment"

📞 Routing call to Alex (Appointments)
   Intent: appointment (confidence: 90%)
📞 Call 9ccce5f3 started with Alex (Appointments)

🤖 Sarah (Sales): Let me connect you with Alex.
🔊 Speech complete

🤖 Alex (Appointments): Hi! I'm Alex. I understand you need help. Let me assist you.
🔊 Speech complete

🤖 Alex (Appointments): I can help you schedule an appointment. 

To accommodate your schedule, I have the following available time slots this week:
- Monday at 10 AM
- Monday at 2 PM
- Tuesday at 9 AM

Which time slot works best for you?
🔊 Speech complete

💬 Turn 2/10
🎤 Listening... 

: 

## Troubleshooting: Test Microphone & Speech Recognition

In [23]:
# Diagnostic tool to test microphone and speech recognition
import speech_recognition as sr

def test_microphone_setup():
    """Test microphone and speech recognition setup"""
    print("="*70)
    print("🔧 MICROPHONE & SPEECH RECOGNITION DIAGNOSTICS")
    print("="*70)
    
    # List available microphones
    print("\n📋 Available microphones:")
    for index, name in enumerate(sr.Microphone.list_microphone_names()):
        print(f"   [{index}] {name}")
    
    # Test default microphone
    print("\n🎤 Testing default microphone...")
    recognizer = sr.Recognizer()
    
    try:
        with sr.Microphone() as source:
            print("✅ Microphone initialized successfully")
            print(f"   Current energy threshold: {recognizer.energy_threshold}")
            
            # Calibrate
            print("\n🔧 Calibrating for ambient noise (3 seconds of silence)...")
            recognizer.adjust_for_ambient_noise(source, duration=3)
            print(f"   Adjusted energy threshold: {recognizer.energy_threshold}")
            
            # Adjust settings for better recognition
            recognizer.dynamic_energy_threshold = True
            recognizer.pause_threshold = 1.2  # Slightly longer pause detection
            recognizer.phrase_threshold = 0.3
            recognizer.non_speaking_duration = 1.0
            
            print("\n🎙️ Please speak a test phrase (you have 10 seconds)...")
            print("   Try saying: 'Hello, this is a test'")
            
            audio = recognizer.listen(source, timeout=10, phrase_time_limit=15)
            print("✅ Audio captured successfully")
            
            print("\n🔄 Sending to Google Speech Recognition...")
            text = recognizer.recognize_google(audio, language='en-US')
            
            print(f"\n✅ SUCCESS! Recognized text:")
            print(f"   \"{text}\"")
            
            return True
            
    except sr.WaitTimeoutError:
        print("\n❌ ERROR: No speech detected within timeout period")
        print("💡 Solutions:")
        print("   - Make sure your microphone is connected and working")
        print("   - Check Windows microphone permissions")
        print("   - Speak louder and closer to the microphone")
        return False
        
    except sr.UnknownValueError:
        print("\n❌ ERROR: Could not understand the audio")
        print("💡 Solutions:")
        print("   - Speak more clearly and slowly")
        print("   - Reduce background noise")
        print("   - Check microphone volume in Windows settings")
        print("   - Try a different microphone")
        return False
        
    except sr.RequestError as e:
        print(f"\n❌ ERROR: Could not request results from Google Speech Recognition")
        print(f"   Error details: {e}")
        print("💡 Solutions:")
        print("   - Check your internet connection")
        print("   - Google Speech API might be temporarily unavailable")
        return False
        
    except Exception as e:
        print(f"\n❌ UNEXPECTED ERROR: {type(e).__name__}: {e}")
        print("💡 Solutions:")
        print("   - Make sure pyaudio is properly installed")
        print("   - Check Windows microphone permissions")
        print("   - Try restarting the Jupyter kernel")
        return False

# Run diagnostics
print("🔍 Running speech recognition diagnostics...\n")
test_microphone_setup()

print("\n" + "="*70)
print("If the test succeeded, your microphone is working correctly!")
print("If it failed, follow the solutions above to fix the issue.")
print("="*70)

🔍 Running speech recognition diagnostics...

🔧 MICROPHONE & SPEECH RECOGNITION DIAGNOSTICS

📋 Available microphones:
   [0] Microsoft Sound Mapper - Input
   [1] Microphone Array (Realtek(R) Au
   [2] Microphone (2- USB Audio Device
   [3] AI Noise-cancelling Input (ASUS
   [4] Microsoft Sound Mapper - Output
   [5] Speakers (Realtek(R) Audio)
   [6] LG ULTRAGEAR (NVIDIA High Defin
   [7] AI Noise-cancelling Output (ASU
   [8] Headphones (Realtek(R) Audio)
   [9] Speakers (2- USB Audio Device)
   [10] Primary Sound Capture Driver
   [11] Microphone Array (Realtek(R) Audio)
   [12] Microphone (2- USB Audio Device)
   [13] AI Noise-cancelling Input (ASUS Utility)
   [14] Primary Sound Driver
   [15] Speakers (Realtek(R) Audio)
   [16] LG ULTRAGEAR (NVIDIA High Definition Audio)
   [17] AI Noise-cancelling Output (ASUS Utility)
   [18] Headphones (Realtek(R) Audio)
   [19] Speakers (2- USB Audio Device)
   [20] LG ULTRAGEAR (NVIDIA High Definition Audio)
   [21] AI Noise-cancelling Output